In [ ]:
#This is pyenv file for making AI classification.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()
import time

In [2]:
data=pd.read_csv('data2010.csv')

In [4]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [5]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
4968,165536,21V050000,CHEVROLET,SUBURBAN,2010,N212328760,AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE,"General Motors, LLC",NaN,NaN,...,"General Motors, LLC (GM) is recalling certain ...",An inflator explosion may result in sharp meta...,"GM will notify owners, and dealers will replac...",Owners may also contact the National Highway T...,108450001450776000001463,MODULE ASM-INFL RST I/P,"Passenger Airbag Module (dual stage, 2011-2014)","22744316, 22744317, 22744318, 22744319, 227443...",No,No
4411,136970,18V024000,TOYOTA,COROLLA,2010,See attached report,AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE,Toyota Motor Engineering & Manufacturing,20071123.0,20101206.0,...,Toyota Motor Engineering & Manufacturing (Toyo...,An inflator explosion may result in sharp meta...,"Toyota will notify the Toyota, Scion and Lexus...",Owners may also contact the National Highway T...,81046000695829000001463,NaN,NaN,NaN,No,No
1042,88597,11V343000,INDIAN,VINTAGE,2010,NaN,EXTERIOR LIGHTING:HEADLIGHTS:SWITCH,INDIAN MOTORCYCLE COMPANY,20081210.0,20110408.0,...,POLARIS IS RECALLING CERTAIN MODEL YEAR 2009-2...,THE BCM TURNS OFF THE HEADLIGHT UNTIL THE NEXT...,DEALERS WILL REPLACE THE BCM FREE OF CHARGE. ...,OWNERS MAY ALSO CONTACT THE NATIONAL HIGHWAY T...,42321001239770000000219,NaN,NaN,NaN,No,No
6116,254710,23V067000,NISSAN,TITAN,2010,"PC944, PC945, PC946",AIR BAGS:FRONTAL,"Nissan North America, Inc.",NaN,NaN,...,"Nissan North America, Inc. (Nissan) is recalli...",A detached emblem can become a projectile duri...,Dealers will replace the drivers airbag cover...,Owners may also contact the National Highway T...,124711000767918000000257,Frontier – Graphite / Red Interior,Driver’s Airbag Assembly,98510-ZP60C,No,No
2874,105852,14V780000,IC,CE,2010,NaN,EQUIPMENT ADAPTIVE/MOBILITY,"Kingmor Supply, Inc.",20060101.0,20141031.0,...,"Kingmor Supply, Inc. (Kingmor) is recalling ce...","If the platform side plates crack, the lift pl...",Kingmor will notify owners and Ricon will supp...,Owners may also contact the National Highway T...,57143001979640000001004,NaN,NaN,NaN,No,No


In [6]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE"))

In [7]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [9]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [10]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-mini'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-mini):  


In [11]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [12]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $1.14'

In [13]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

def classify_fix(prompt_including_tweet):
    for attempt in range(3):
        try:
            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result
        except Exception as e:
            time.sleep(attempt*3)
    return None    

In [14]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify_fix)

100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [01:03<00:00,  1.26s/it]


In [15]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
4968,"General Motors, LLC (GM) is recalling certain 2007-2011 Cadillac Escalade, Escalade ESV, Escalade EXT, Chevrolet Avalanche, Silverado 1500, Suburban, Tahoe, GMC Sierra 1500, Yukon, Yukon XL, and 2009-2011 Chevrolet Silverado 2500, Silverado 3500, GMC Sierra 2500 and Sierra 3500 vehicles originally sold, or ever registered, in the states of Alabama, California, Florida, Georgia, Hawaii, Louisiana, Mississippi, South Carolina, Texas, Puerto Rico, American Samoa, Guam, the Northern Mariana Isla...",safety_equipment
4411,"Toyota Motor Engineering & Manufacturing (Toyota) is recalling certain 2013 Scion xB, Toyota Corolla, Corolla Matrix, Sienna, 4Runner, Lexus IS250, IS350, IS250C, IS350C, IS-F, and GX460 vehicles sold, ever registered, in the states of AL, CA, FL, GA, HI, LA, MS, SC, TX, PR, AS, GU, the MP (Saipan), and the U.S. VI or ""Zone A."" Toyota is also recalling certain 2010 Scion xB, Toyota Corolla, Corolla Matrix, Yaris, 4Runner, Lexus IS250, IS350, IS250C, IS350C, ES350, IS-F, GX460, and Pontiac...",safety_equipment
1042,"POLARIS IS RECALLING CERTAIN MODEL YEAR 2009-2011 INDIAN MOTORCYCLES FOR THE BODY CONTROL MODULE (BCM) WHICH CAN CAUSE THE HEADLIGHT TO TURN OFF IN SOME CIRCUMSTANCES. IF THE HEADLIGHT SWITCH IS SLOWLY MOVED FROM ONE POSITION TO ANOTHER (HIGH BEAM TO LOW BEAM OR LOW BEAM TO HIGH BEAM), BOTH BEAMS MAY BE ON MOMENTARILY AND THE BCM INTERPRETS THIS AS AN OVERLOAD CONDITION.",electronics
6116,"Nissan North America, Inc. (Nissan) is recalling certain 2008-2011 Titan, Frontier, Xterra, Pathfinder, Armada, and 2008-2009 Quest vehicles. The resin emblem on the driver's air bag cover may develop cracks and detach during air bag deployment.",safety_equipment
2874,"Kingmor Supply, Inc. (Kingmor) is recalling certain model year 2006-2014 IC CE school buses manufactured from January 1, 2006, to October 31, 2014, and equipped with certain model S2005, S2010, S5005, S5010, S5505, and S5510 wheelchair lifts manufactured by Ricon Corporation. The platform side plate of the affected wheelchair lifts may crack.",safety_equipment
5514,"Ford Motor Company (Ford) is recalling certain 2005-2014 Mustang, 2004-2011 Ranger, 2009-2011 Mercury Milan, 2006 Ford GT, 2008-2012 Fusion, 2007-2010 Edge and Lincoln MKX, and 2009-2010 Lincoln MKZ vehicles. These vehicles may be equipped with a driver or passenger frontal air bag inflator used as a replacement service part, that may explode due to propellant degradation occurring after long-term exposure to high absolute humidity, temperature and temperature cycling.",safety_equipment
2171,"Circle D Corporation, Inc. (Circle D) is recalling certain model year 2006-2013 BHST, GNST, and GNMPT horse and multipurpose trailers with tack compartments, manufactured January 1, 2006, through April 25, 2013, and equipped with a Fastec-brand door lock. It may be possible to remove the key from the lock in a way that potentially jams the deadbolt.",safety_equipment
6231,"PACCAR Incorporated (PACCAR) is recalling certain 2009-2021 Kenworth C500, 2012-2021 Kenworth T440, 2009-2015 Kenworth T660, 2015-2024 Kenworth T680, 2009-2024 Kenworth T800, 2009-2020 Kenworth T370, 2014-2023 Kenworth T880, 2011-2016 Kenworth T470, 2023 Kenworth T480, and 2009-2019 Kenworth W900 vehicles. The air tanks have an insufficient volume of air for the brake system. As such, these vehicles fail to comply with the requirements of Federal Motor Vehicle Safety Standard number 121, ""...",safety_equipment
4296,Altec Industries Inc. (Altec) is recalling certain 2010-2011 Effer 440 and 470 Knuckle Boom Cranes. The inner race rotation bearing fasteners for the crane turret may fail and the pedestal may crack if the fasteners are not replaced every two years.,others
5534,"Ford Motor Company (Ford) is recalling certain 2005-2014 Mustang, 2004-2011 Ranger, 2009-2011 Mercury Milan, 2006 Ford GT, 2008-2012 Fusion, 2007-2010 Edge and Lincoln MKX, and 2009-2010 Lincoln MKZ vehicles. These vehicles may be equ

In [16]:
okay_to_do_the_whole_thing = input()

In [17]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [18]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify_fix)
data.to_csv(data_dir + 'Recall_classified2010.csv')

100%|██████████████████████████████████████████████████████████████████████████████| 6587/6587 [2:22:52<00:00,  1.30s/it]


In [3]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-fa3...4da",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 14.53097225,
    "include_byok_in_limit": false,
    "usage": 32.5680895,
    "usage_daily": 5.46902775,
    "usage_weekly": 26.90956805,
    "usage_monthly": 30.6411356,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}
